# Experiment 8: Time Series Forecasting

**Aim:** Visualize moving average, trend and three future monthly predictions.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default='notebook_connected'
DATA=Path('data') if Path('data').exists() else Path('DV/data')
df=pd.read_csv(DATA/'retail_sales.csv',parse_dates=['Date'])

import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
monthly=df.set_index('Date').Sales.resample('MS').sum().reset_index()
monthly['MovingAverage']=monthly.Sales.rolling(3).mean()
X=np.arange(len(monthly)).reshape(-1,1)
model=LinearRegression().fit(X,monthly.Sales)
monthly['Trend']=model.predict(X)
dates=pd.date_range(monthly.Date.iloc[-1]+pd.offsets.MonthBegin(1),periods=3,freq='MS')
forecast=model.predict(np.arange(len(monthly),len(monthly)+3).reshape(-1,1))
print(pd.DataFrame({'Date':dates,'Forecast':forecast}).to_string(index=False))
fig=go.Figure()
for col in ['Sales','MovingAverage','Trend']:
    fig.add_scatter(x=monthly.Date,y=monthly[col],mode='lines',name=col)
fig.add_scatter(x=[monthly.Date.iloc[-1],*dates],y=[monthly.Trend.iloc[-1],*forecast],mode='lines+markers',line=dict(dash='dash'),name='Forecast')
fig.update_layout(title='Monthly sales: trend and forecast'); fig.show()

      Date     Forecast
2026-01-01 29826.286429
2026-02-01 30034.235920
2026-03-01 30242.185412


Monthly sales: trend and forecast

## Power BI Desktop procedure
1. Import retail_sales.csv. Create column `MonthStart = DATE(YEAR(retail_sales[Date]), MONTH(retail_sales[Date]), 1)` and measure `Total Sales = SUM(retail_sales[Sales])`.
2. Add line chart with MonthStart on continuous X-axis and Total Sales on Y-axis.
3. Add moving-average measure:
   ```dax
   Moving Average 3M =
   VAR LastMonth = MAX(retail_sales[MonthStart])
   VAR Months = FILTER(ALL(retail_sales[MonthStart]), retail_sales[MonthStart] <= LastMonth && retail_sales[MonthStart] >= EDATE(LastMonth, -2))
   RETURN IF(COUNTROWS(Months) = 3, AVERAGEX(Months, CALCULATE([Total Sales])))
   ```
4. Plot moving average alongside Total Sales. Duplicate chart and keep only Total Sales; in Analytics add Forecast for three months if available. Add trend line in a supported visual. The Python forecast is illustrative linear extrapolation, not validated predictions.